# Seasonal Threshold Day Count Check

This notebook runs seasonal threshold-day generation (e.g. dry days in summer/winter) and performs basic NetCDF sanity checks.

In [1]:
import os
from pathlib import Path

import numpy as np
import xarray as xr
import yaml

import sys
sys.path.append(str(Path().resolve().parent / 'src'))  # Add data/src to sys.path

from process_daily_data import ClimateDataProcessor

In [2]:
# Run from repository root: research.LCAT.public
config_filepath = Path().resolve().parent / 'config.yml'
with open(config_filepath) as f:
    config = yaml.safe_load(f)

processor = ClimateDataProcessor(config, ensemble_member=1)
print('Data root:', config['chess_scape_netcdf_location'])

Data root: /home/cat/Desktop/projects/lcat/research.LCAT.public/data/store/chess-scape


## Generate seasonal day-count files

This is a minimal run for two metrics: `dry_days` (from `pr`) and `tropical_nights` (from `tasmin`).

In [ ]:
processor.generate_data(
    quantiles_config={},
    tropical_nights_enabled=True,
    hot_days_enabled=False,
    heavy_rain_enabled=False,
    dry_days_enabled=True,
    windy_days_enabled=False,
    rcps=[60],
    bias_options=[True],
    seasons=['summer', 'winter'],
    variables=['pr', 'tasmin'],
    tropical_threshold=20.0,
    dry_threshold=1.0,
)

Skipping processing for variable pr (no outputs requested).
Processing RCP 60, Bias Corrected: True, Variable: tasmin
Getting grid dimensions...
Getting grid dimensions...
Grid size: 1057 x 656
Filtering files for season: summer
Files after season filtering: 300 out of 1200
Files per decade: [(0, 30), (9, 30)]

Processing step-decade 0 (30 files)...
Grid size: 1057 x 656
Filtering files for season: summer
Files after season filtering: 300 out of 1200
Files per decade: [(0, 30), (9, 30)]

Processing step-decade 0 (30 files)...


Loading files: 100%|██████████| 30/30 [01:39<00:00,  3.31s/it]



Successfully processed 30/30 files
Step-decade 0 data shape: (900, 1057, 656)
Applying calculation for step-decade 0...
Step-decade 0 data shape: (900, 1057, 656)
Applying calculation for step-decade 0...
Processing 900 days (10.0 summer periods) of data
Threshold: 20.0, Comparison: gte, Season: summer
Processing 900 days (10.0 summer periods) of data
Threshold: 20.0, Comparison: gte, Season: summer

Processing step-decade 9 (30 files)...

Processing step-decade 9 (30 files)...


Loading files: 100%|██████████| 30/30 [01:42<00:00,  3.41s/it]



Successfully processed 30/30 files
Step-decade 9 data shape: (900, 1057, 656)
Applying calculation for step-decade 9...
Step-decade 9 data shape: (900, 1057, 656)
Applying calculation for step-decade 9...
Processing 900 days (10.0 summer periods) of data
Threshold: 20.0, Comparison: gte, Season: summer
Processing 900 days (10.0 summer periods) of data
Threshold: 20.0, Comparison: gte, Season: summer
Processing Complete
Saved dataset to /home/cat/Desktop/projects/lcat/research.LCAT.public/data/store/chess-scape/data/rcp60_bias-corrected/01/seasonal/chess-scape_rcp60_bias-corrected_01_tropical_nights_uk_1km_summer_19801201-20801130.nc
Skipping processing for variable pr (no outputs requested).
Processing RCP 60, Bias Corrected: True, Variable: tasmin
Processing Complete
Saved dataset to /home/cat/Desktop/projects/lcat/research.LCAT.public/data/store/chess-scape/data/rcp60_bias-corrected/01/seasonal/chess-scape_rcp60_bias-corrected_01_tropical_nights_uk_1km_summer_19801201-20801130.nc
Ski

Loading files: 100%|██████████| 30/30 [01:45<00:00,  3.52s/it]



Successfully processed 30/30 files
Step-decade 0 data shape: (900, 1057, 656)
Applying calculation for step-decade 0...
Step-decade 0 data shape: (900, 1057, 656)
Applying calculation for step-decade 0...
Processing 900 days (10.0 winter periods) of data
Threshold: 20.0, Comparison: gte, Season: winter
Processing 900 days (10.0 winter periods) of data
Threshold: 20.0, Comparison: gte, Season: winter

Processing step-decade 9 (30 files)...

Processing step-decade 9 (30 files)...


Loading files: 100%|██████████| 30/30 [01:52<00:00,  3.75s/it]



Successfully processed 30/30 files
Step-decade 9 data shape: (900, 1057, 656)
Applying calculation for step-decade 9...
Step-decade 9 data shape: (900, 1057, 656)
Applying calculation for step-decade 9...
Processing 900 days (10.0 winter periods) of data
Threshold: 20.0, Comparison: gte, Season: winter
Processing 900 days (10.0 winter periods) of data
Threshold: 20.0, Comparison: gte, Season: winter
Processing Complete
Saved dataset to /home/cat/Desktop/projects/lcat/research.LCAT.public/data/store/chess-scape/data/rcp60_bias-corrected/01/seasonal/chess-scape_rcp60_bias-corrected_01_tropical_nights_uk_1km_winter_19801201-20801130.nc

SUMMARY: 2 datasets saved
  /home/cat/Desktop/projects/lcat/research.LCAT.public/data/store/chess-scape/data/rcp60_bias-corrected/01/seasonal/chess-scape_rcp60_bias-corrected_01_tropical_nights_uk_1km_summer_19801201-20801130.nc
  /home/cat/Desktop/projects/lcat/research.LCAT.public/data/store/chess-scape/data/rcp60_bias-corrected/01/seasonal/chess-scape_r

In [4]:
def expected_output_path(config_dict, rcp, bias_corrected, ensemble_member, metric, season):
    base = Path(config_dict['chess_scape_netcdf_location'])
    bias_suffix = '_bias-corrected' if bias_corrected else ''
    season_folder = 'annual' if season == 'annual' else 'seasonal'
    ensemble_str = f'{ensemble_member:02d}'
    filename = (
        f'chess-scape_rcp{rcp}{bias_suffix}_{ensemble_str}_{metric}_'
        f'uk_1km_{season}_19801201-20801130.nc'
    )
    return base / f'data/rcp{rcp}{bias_suffix}/{ensemble_str}/{season_folder}' / filename

summer_dry_file = expected_output_path(config, 60, True, 1, 'dry_days', 'summer')
winter_dry_file = expected_output_path(config, 60, True, 1, 'dry_days', 'winter')
summer_tropical_file = expected_output_path(config, 60, True, 1, 'tropical_nights', 'summer')
winter_tropical_file = expected_output_path(config, 60, True, 1, 'tropical_nights', 'winter')

for label, path in [
    ('summer_dry', summer_dry_file),
    ('winter_dry', winter_dry_file),
    ('summer_tropical', summer_tropical_file),
    ('winter_tropical', winter_tropical_file),
]:
    print(label, path)
    print('  exists:', path.exists())

summer_dry /home/cat/Desktop/projects/lcat/research.LCAT.public/data/store/chess-scape/data/rcp60_bias-corrected/01/seasonal/chess-scape_rcp60_bias-corrected_01_dry_days_uk_1km_summer_19801201-20801130.nc
  exists: True
winter_dry /home/cat/Desktop/projects/lcat/research.LCAT.public/data/store/chess-scape/data/rcp60_bias-corrected/01/seasonal/chess-scape_rcp60_bias-corrected_01_dry_days_uk_1km_winter_19801201-20801130.nc
  exists: True
summer_tropical /home/cat/Desktop/projects/lcat/research.LCAT.public/data/store/chess-scape/data/rcp60_bias-corrected/01/seasonal/chess-scape_rcp60_bias-corrected_01_tropical_nights_uk_1km_summer_19801201-20801130.nc
  exists: True
winter_tropical /home/cat/Desktop/projects/lcat/research.LCAT.public/data/store/chess-scape/data/rcp60_bias-corrected/01/seasonal/chess-scape_rcp60_bias-corrected_01_tropical_nights_uk_1km_winter_19801201-20801130.nc
  exists: True


In [5]:
def inspect_day_count_file(path, season, metric_name):
    ds = xr.open_dataset(path, engine='netcdf4')
    try:
        data = ds['variable']
        arr = data.values

        print('---', metric_name, '---')
        print('file:', path.name)
        print('dims:', data.dims)
        print('shape:', arr.shape)
        print('decades:', ds.decade.values)
        print('min:', float(np.nanmin(arr)))
        print('mean:', float(np.nanmean(arr)))
        print('max:', float(np.nanmax(arr)))

        # Seasonal means should be bounded by days in that season (360-day calendar => 90 days)
        upper_bound = 360 if season == 'annual' else 90
        assert float(np.nanmax(arr)) <= upper_bound + 1e-6, (
            f'Max value exceeds expected upper bound of {upper_bound} days'
        )
    finally:
        ds.close()

inspect_day_count_file(summer_dry_file, 'summer', 'dry_days_summer')
inspect_day_count_file(winter_dry_file, 'winter', 'dry_days_winter')
inspect_day_count_file(summer_tropical_file, 'summer', 'tropical_nights_summer')
inspect_day_count_file(winter_tropical_file, 'winter', 'tropical_nights_winter')

--- dry_days_summer ---
file: chess-scape_rcp60_bias-corrected_01_dry_days_uk_1km_summer_19801201-20801130.nc
dims: ('decade', 'y', 'x')
shape: (2, 1057, 656)
decades: [0 9]
min: 0.0
mean: 18.732587915637897
max: 72.9
--- dry_days_winter ---
file: chess-scape_rcp60_bias-corrected_01_dry_days_uk_1km_winter_19801201-20801130.nc
dims: ('decade', 'y', 'x')
shape: (2, 1057, 656)
decades: [0 9]
min: 0.0
mean: 14.334348247455985
max: 60.3
--- tropical_nights_summer ---
file: chess-scape_rcp60_bias-corrected_01_tropical_nights_uk_1km_summer_19801201-20801130.nc
dims: ('decade', 'y', 'x')
shape: (2, 1057, 656)
decades: [0 9]
min: 0.0
mean: 0.16198542815607908
max: 28.9
--- tropical_nights_winter ---
file: chess-scape_rcp60_bias-corrected_01_tropical_nights_uk_1km_winter_19801201-20801130.nc
dims: ('decade', 'y', 'x')
shape: (2, 1057, 656)
decades: [0 9]
min: 0.0
mean: 0.0
max: 0.0


If the assertions pass and values look plausible (max below 90 for seasonal), the seasonal threshold-day flow is working for generated NetCDF outputs.